# Vertex AI Gemini 3.0 Flash / Pro with Google AI Search Test

## Need to brew install google-cloud-sdk --> gcloud auth application-default login first
Testing Gemini 3.0 Flash with Google AI search embedded via Vertex AI API.


In [17]:
%pip install google-genai python-dotenv pillow requests google-cloud-aiplatform

from google import genai
from google.genai.types import HttpOptions, Part, GenerateContentConfig, Blob
import os
from dotenv import load_dotenv
from PIL import Image
import requests
from io import BytesIO
import base64

load_dotenv()


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 34.6 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 27.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10/10 [google-cloud-aiplatform]-cloud-aiplatform]ager]

[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


True

In [12]:
# Vertex AI Configuration
# For Vertex AI, you need OAuth2 authentication, not API keys
# Set up authentication using one of these methods:
# 1. gcloud auth application-default login
# 2. Set GOOGLE_APPLICATION_CREDENTIALS to service account JSON path
# 3. Set GCP_PROJECT_ID and GCP_LOCATION environment variables

project_id = os.getenv("GCP_PROJECT_ID")
location = os.getenv("GCP_LOCATION", "us-central1")
model_name = "gemini-2.5-flash"

print(f"Project ID: {project_id or 'Not set (will use default)'}")
print(f"Location: {location}")
print(f"Model: {model_name}")

Success!


In [ ]:
# Initialize client - Using Google AI API with API key (no gcloud needed!)
api_key = os.getenv("GOOGLE_AI_API_KEY") or os.getenv("VERTEX_AI_API_KEY")
project_id = os.getenv("GCP_PROJECT_ID")  # Optional, for Live API

client = None
client_type = None

if api_key:
    try:
        # Use Google AI API with API key (works without gcloud!)
        # For Live API, you may need project parameter
        if project_id:
            client = genai.Client(api_key=api_key, project=project_id)
        else:
            client = genai.Client(api_key=api_key)
        client_type = "google_ai"
        print("✅ Google AI client initialized (using API key)")
        print(f"Model: {model_name}")
        if project_id:
            print(f"Project: {project_id} (for Live API)")
        print("\n💡 No gcloud needed! Using API key authentication.")
    except Exception as e:
        print(f"❌ Google AI API init failed: {e}")
        client = None
        import traceback
        traceback.print_exc()

if not client:
    print("\n❌ Failed to initialize client")
    print("\nPlease set your API key:")
    print("  export GOOGLE_AI_API_KEY=your_api_key")
    print("\nOr add to .env file:")
    print("  GOOGLE_AI_API_KEY=your_api_key")
    print("\nNote: For Live API, you may also need:")
    print("  export GCP_PROJECT_ID=your_project_id")

✅ Google AI client initialized (using API key)
Model: gemini-2.5-flash

💡 No gcloud needed! Using API key authentication.


In [15]:
# Simple test prompt
prompt = "Say 'Gemini 2.5 Flash is working!' and explain what you're best at."

print(f"Prompt: {prompt}")
print("\n" + "="*70)


Prompt: Say 'Gemini 2.5 Flash is working!' and explain what you're best at.



## Multimodal Capabilities

Gemini 2.5 Flash supports multimodal inputs including images, text, and more. Let's test image analysis capabilities.


In [21]:
# Test 1: Image analysis from URL using fileUri
image_uri = "gs://cloud-samples-data/generative-ai/image/scones.jpg"
image_prompt = "What's in this image? Describe it in detail."

print("Testing image analysis from Cloud Storage URI...")
print(f"Image URI: {image_uri}")
print(f"Prompt: {image_prompt}")
print("\n" + "="*70)

if client:
    try:
        # Use Part.from_uri for Cloud Storage or HTTP URLs
        response = client.models.generate_content(
            model=model_name,
            contents=[
                image_prompt,
                Part.from_uri(
                    file_uri=image_uri,
                    mime_type="image/jpeg"
                )
            ],
            config=GenerateContentConfig(
                temperature=0.7,
                max_output_tokens=500,
            )
        )
        
        print("✅ Response:")
        print(response.text)
        
    except Exception as e:
        print(f"❌ Error: {e}")
        print("\nTrying with HTTP URL instead...")
        try:
            # Fallback to HTTP URL
            http_url = "https://storage.googleapis.com/generativeai-downloads/images/scones.jpg"
            response = client.models.generate_content(
                model=model_name,
                contents=[
                    image_prompt,
                    Part.from_uri(
                        file_uri=http_url,
                        mime_type="image/jpeg"
                    )
                ],
                config=GenerateContentConfig(
                    temperature=0.7,
                    max_output_tokens=500,
                )
            )
            print("✅ Response (using HTTP URL):")
            print(response.text)
        except Exception as e2:
            print(f"❌ Fallback error: {e2}")
            import traceback
            traceback.print_exc()
else:
    print("⚠️  Client not initialized. Check authentication.")


Testing image analysis from Cloud Storage URI...
Image URI: gs://cloud-samples-data/generative-ai/image/scones.jpg
Prompt: What's in this image? Describe it in detail.

❌ Error: 401 UNAUTHENTICATED. {'error': {'code': 401, 'message': 'API keys are not supported by this API. Expected OAuth2 access token or other authentication credentials that assert a principal. See https://cloud.google.com/docs/authentication', 'status': 'UNAUTHENTICATED', 'details': [{'@type': 'type.googleapis.com/google.rpc.ErrorInfo', 'reason': 'CREDENTIALS_MISSING', 'domain': 'googleapis.com', 'metadata': {'method': 'google.ai.generativelanguage.v1beta.GenerativeService.GenerateContent', 'service': 'generativelanguage.googleapis.com'}}]}}

Trying with HTTP URL instead...
❌ Fallback error: 401 UNAUTHENTICATED. {'error': {'code': 401, 'message': 'API keys are not supported by this API. Expected OAuth2 access token or other authentication credentials that assert a principal. See https://cloud.google.com/docs/authenti

Traceback (most recent call last):
  File "/var/folders/zr/w_kl9yzd6hsbts7q37qt6d1r0000gn/T/ipykernel_47254/3786166392.py", line 13, in <module>
    response = client.models.generate_content(
        model=model_name,
    ...<10 lines>...
        )
    )
  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/google/genai/models.py", line 5056, in generate_content
    response = self._generate_content(
        model=model, contents=contents, config=parsed_config
    )
  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/google/genai/models.py", line 3843, in _generate_content
    response = self._api_client.request(
        'post', path, request_dict, http_options
    )
  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/google/genai/_api_client.py", line 1331, in request
    response = self._request(http_request, http_options, stream=False)
  File "/Library/Frameworks/Python.framewor

In [ ]:
# Test 2: Image analysis from local file using inline data
local_image_path = None  # Set to your image path, e.g., "path/to/image.jpg"

if local_image_path and os.path.exists(local_image_path):
    print("Testing image analysis from local file...")
    print(f"Image path: {local_image_path}")
    print("\n" + "="*70)
    
    if client:
        try:
            # Read and encode image as base64
            with open(local_image_path, "rb") as f:
                image_data = base64.b64encode(f.read()).decode('utf-8')
            
            # Determine MIME type
            mime_type = "image/jpeg"
            if local_image_path.lower().endswith('.png'):
                mime_type = "image/png"
            elif local_image_path.lower().endswith('.gif'):
                mime_type = "image/gif"
            elif local_image_path.lower().endswith('.webp'):
                mime_type = "image/webp"
            
            # Use inline data with Blob
            # Note: For local files, you can also upload to GCS first and use fileUri
            inline_data = Blob(
                mime_type=mime_type,
                data=image_data  # base64 encoded string
            )
            
            # Create Part with inline_data
            # Alternative: Upload to GCS and use Part.from_uri() instead
            image_part = Part(inline_data=inline_data)
            
            response = client.models.generate_content(
                model=model_name,
                contents=[
                    "Describe this image in detail. What do you see?",
                    image_part
                ],
                config=GenerateContentConfig(
                    temperature=0.7,
                    max_output_tokens=500,
                )
            )
            
            print("✅ Response:")
            print(response.text)
            
        except Exception as e:
            print(f"❌ Error: {e}")
            import traceback
            traceback.print_exc()
    else:
        print("⚠️  Client not initialized. Check authentication.")
else:
    print("ℹ️  No local image path provided. Set 'local_image_path' variable to test with local images.")
    print("   Example: local_image_path = 'path/to/your/image.jpg'")


In [ ]:
# Test 3: Multiple images with text
image_uris = [
    "gs://cloud-samples-data/generative-ai/image/scones.jpg",
    # Add more URIs if needed
]

multimodal_prompt = """Compare these images. What are the similarities and differences? 
If there's only one image, describe it in detail and suggest what it could be used for."""

print("Testing multiple images with text...")
print(f"Number of images: {len(image_uris)}")
print(f"Prompt: {multimodal_prompt}")
print("\n" + "="*70)

if client and image_uris:
    try:
        # Prepare content with text and images
        contents = [multimodal_prompt]
        for uri in image_uris:
            contents.append(
                Part.from_uri(
                    file_uri=uri,
                    mime_type="image/jpeg"
                )
            )
            print(f"✅ Added image: {uri}")
        
        # Generate content
        response = client.models.generate_content(
            model=model_name,
            contents=contents,
            config=GenerateContentConfig(
                temperature=0.7,
                max_output_tokens=800,
            )
        )
        
        print("\n✅ Response:")
        print(response.text)
            
    except Exception as e:
        print(f"❌ Error: {e}")
        import traceback
        traceback.print_exc()
else:
    if not client:
        print("⚠️  Client not initialized. Check authentication.")
    else:
        print("⚠️  No image URIs provided.")


In [ ]:
# Test 4: Image + Text Q&A
image_uri = "gs://cloud-samples-data/generative-ai/image/scones.jpg"
qa_prompt = """Look at this image and answer:
1. What is the main subject?
2. What colors are prominent?
3. What could this image be used for?
4. Write a creative caption for this image."""

print("Testing image Q&A...")
print(f"Image URI: {image_uri}")
print(f"Prompt: {qa_prompt}")
print("\n" + "="*70)

if client:
    try:
        response = client.models.generate_content(
            model=model_name,
            contents=[
                qa_prompt,
                Part.from_uri(
                    file_uri=image_uri,
                    mime_type="image/jpeg"
                )
            ],
            config=GenerateContentConfig(
                temperature=0.8,  # Slightly higher for creative caption
                max_output_tokens=600,
            )
        )
        
        print("✅ Response:")
        print(response.text)
        
    except Exception as e:
        print(f"❌ Error: {e}")
        import traceback
        traceback.print_exc()
else:
    print("⚠️  Client not initialized. Check authentication.")


In [ ]:
# Test 5: Image analysis with specific task (e.g., OCR, object detection)
image_uri = "gs://cloud-samples-data/generative-ai/image/scones.jpg"
task_prompt = """Analyze this image and provide:
- A detailed description
- Any text visible in the image (OCR)
- Objects or items you can identify
- The style or type of photography
- Any interesting details"""

print("Testing detailed image analysis...")
print(f"Image URI: {image_uri}")
print("\n" + "="*70)

if client:
    try:
        response = client.models.generate_content(
            model=model_name,
            contents=[
                task_prompt,
                Part.from_uri(
                    file_uri=image_uri,
                    mime_type="image/jpeg"
                )
            ],
            config=GenerateContentConfig(
                temperature=0.5,  # Lower temperature for more factual analysis
                max_output_tokens=700,
            )
        )
        
        print("✅ Response:")
        print(response.text)
        
    except Exception as e:
        print(f"❌ Error: {e}")
        import traceback
        traceback.print_exc()
else:
    print("⚠️  Client not initialized. Check authentication.")


## Live API with Native Audio

Gemini 2.5 Flash supports real-time bidirectional audio streaming via the Live API. This is a preview feature that enables natural voice conversations.


In [19]:
# Test 1: Text-based Live API session (simpler test)
# This tests the Live API without audio first

live_model = "gemini-2.5-flash-preview-native-audio-09-2025"

print("Testing Live API with text input...")
print(f"Model: {live_model}")
print("\n" + "="*70)

if client:
    try:
        from google.genai import types
        
        # Create a Live API session
        session = client.agentic_loop.create_session(
            model=live_model,
            config=types.LiveConnectConfig(
                system_instruction="You are a helpful AI assistant. Speak naturally and concisely.",
            )
        )
        
        print("✅ Live API session created")
        
        # Send a text message
        print("\n📤 Sending: 'Hello! Can you introduce yourself?'")
        session.send(types.Content(
            parts=[types.Part(text="Hello! Can you introduce yourself?")]
        ))
        
        # Get response
        print("\n📥 Response:")
        for response in session.receive():
            if hasattr(response, 'text') and response.text:
                print(response.text)
            elif hasattr(response, 'parts') and response.parts:
                for part in response.parts:
                    if hasattr(part, 'text') and part.text:
                        print(part.text)
            break  # Just get first response for testing
        
        session.close()
        print("\n✅ Session closed")
        
    except Exception as e:
        print(f"❌ Error: {e}")
        print("\nNote: Live API requires:")
        print("  1. Model: gemini-2.5-flash-preview-native-audio-09-2025")
        print("  2. Location: us-central1 (for Vertex AI)")
        print("  3. Preview access enabled")
        import traceback
        traceback.print_exc()
else:
    print("⚠️  Client not initialized. Check authentication.")


Testing Live API with text input...
Model: gemini-2.5-flash-preview-native-audio-09-2025

⚠️  Client not initialized. Check authentication.


In [24]:
# Test 2: Native Audio Live API (requires pyaudio)
# Real-time bidirectional audio streaming

%pip install pyaudio

try:
    import pyaudio
    import threading
    import time
    
    print("✅ pyaudio installed")
    print("\n" + "="*70)
    print("🎤 Native Audio Live API Test")
    print("="*70)
    
    if client:
        try:
            from google.genai import types
            
            live_model = "gemini-2.5-flash-preview-native-audio-09-2025"
            
            # Create session with native audio
            session = client.agentic_loop.create_session(
                model=live_model,
                config=types.LiveConnectConfig(
                    system_instruction="You are a helpful AI assistant. Speak naturally and concisely.",
                )
            )
            
            print("✅ Live Audio Session Started")
            print("⚠️  Press Ctrl+C to stop")
            print("\nNote: This will start recording from your microphone!")
            print("      Make sure you have microphone permissions enabled.")
            
            # Audio configuration
            INPUT_RATE = 16000  # 16kHz required for input
            OUTPUT_RATE = 24000  # 24kHz for output
            CHUNK_SIZE = 1024
            
            p = pyaudio.PyAudio()
            
            # Input stream (microphone)
            input_stream = p.open(
                format=pyaudio.paInt16,
                channels=1,
                rate=INPUT_RATE,
                input=True,
                frames_per_buffer=CHUNK_SIZE
            )
            
            # Output stream (speakers)
            output_stream = p.open(
                format=pyaudio.paInt16,
                channels=1,
                rate=OUTPUT_RATE,
                output=True,
                frames_per_buffer=CHUNK_SIZE
            )
            
            stop_flag = threading.Event()
            
            def audio_input_thread():
                """Capture audio from microphone and send to session"""
                try:
                    while not stop_flag.is_set():
                        data = input_stream.read(CHUNK_SIZE, exception_on_overflow=False)
                        session.send(types.Content(
                            parts=[types.Part(inline_data=types.Blob(
                                mime_type="audio/pcm",
                                data=data
                            ))]
                        ))
                except Exception as e:
                    print(f"❌ Input thread error: {e}")
                finally:
                    input_stream.stop_stream()
                    input_stream.close()
            
            def audio_output_thread():
                """Receive audio responses and play them"""
                try:
                    for response in session.receive():
                        if stop_flag.is_set():
                            break
                        if hasattr(response, 'parts') and response.parts:
                            for part in response.parts:
                                if hasattr(part, 'inline_data') and part.inline_data:
                                    if part.inline_data.mime_type == "audio/pcm":
                                        audio_data = part.inline_data.data
                                        output_stream.write(audio_data)
                except Exception as e:
                    print(f"❌ Output thread error: {e}")
                finally:
                    output_stream.stop_stream()
                    output_stream.close()
                    p.terminate()
            
            # Start threads
            input_thread = threading.Thread(target=audio_input_thread, daemon=True)
            output_thread = threading.Thread(target=audio_output_thread, daemon=True)
            
            input_thread.start()
            output_thread.start()
            
            print("\n🎙️  Recording and streaming... (Press Ctrl+C to stop)")
            
            try:
                # Keep main thread alive
                while True:
                    time.sleep(1)
            except KeyboardInterrupt:
                print("\n✋ Stopping session...")
                stop_flag.set()
                session.close()
                print("✅ Session ended")
                
        except Exception as e:
            print(f"❌ Error: {e}")
            import traceback
            traceback.print_exc()
    else:
        print("⚠️  Client not initialized. Check authentication.")
        
except ImportError:
    print("❌ pyaudio not installed")
    print("\nInstall with:")
    print("  pip install pyaudio")
    print("\nNote: On macOS, you may need:")
    print("  brew install portaudio")
    print("  pip install pyaudio")
except Exception as e:
    print(f"❌ Error setting up audio: {e}")
    import traceback
    traceback.print_exc()


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for pyaudio: filename=pyaudio-0.2.14-cp313-cp313-macosx_10_13_universal2.whl size=39427 sha256=da551e8487e92f0a7565c0a944429c414609d03e955b384ae51668236d464900
  Stored in directory: /Users/kc/Library/Caches/pip/wheels/32/45/57/aac45d8ad6f62e05779c15d0e62a09cfb51ff47c3fb1599b36
Successfully built pyaudio

[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
✅ pyaudio installed

🎤 Native Audio Live API Test
❌ Error: 'Client' object has no attribute 'agentic_loop'


Traceback (most recent call last):
  File "/var/folders/zr/w_kl9yzd6hsbts7q37qt6d1r0000gn/T/ipykernel_47254/134754596.py", line 23, in <module>
    session = client.agentic_loop.create_session(
              ^^^^^^^^^^^^^^^^^^^
AttributeError: 'Client' object has no attribute 'agentic_loop'


In [23]:
# Test 3: Live API with Google Search Grounding
# Add search capabilities to Live API sessions

live_model = "gemini-2.5-flash-preview-native-audio-09-2025"

print("Testing Live API with Google Search grounding...")
print(f"Model: {live_model}")
print("\n" + "="*70)

if client:
    try:
        from google.genai import types
        
        # Create session with grounding enabled
        session = client.agentic_loop.create_session(
            model=live_model,
            config=types.LiveConnectConfig(
                system_instruction="You are a helpful AI assistant with access to real-time search.",
                # Enable grounding with Google Search
                tools=[types.Tool(google_search={})] if hasattr(types, 'Tool') else None
            )
        )
        
        print("✅ Live API session with search created")
        
        # Send a query that benefits from search
        query = "What are the latest AI developments in January 2025?"
        print(f"\n📤 Sending: '{query}'")
        session.send(types.Content(
            parts=[types.Part(text=query)]
        ))
        
        # Get response
        print("\n📥 Response:")
        for response in session.receive():
            if hasattr(response, 'text') and response.text:
                print(response.text)
            elif hasattr(response, 'parts') and response.parts:
                for part in response.parts:
                    if hasattr(part, 'text') and part.text:
                        print(part.text)
            
            # Check for citations
            if hasattr(response, 'citation_metadata') and response.citation_metadata:
                print("\n📚 Citations:")
                for citation in response.citation_metadata.citations:
                    print(f"  - {citation.title or citation.uri}")
            break
        
        session.close()
        print("\n✅ Session closed")
        
    except Exception as e:
        print(f"❌ Error: {e}")
        print("\nNote: Search grounding may require additional configuration")
        import traceback
        traceback.print_exc()
else:
    print("⚠️  Client not initialized. Check authentication.")


Testing Live API with Google Search grounding...
Model: gemini-2.5-flash-preview-native-audio-09-2025

❌ Error: 'Client' object has no attribute 'agentic_loop'

Note: Search grounding may require additional configuration


Traceback (most recent call last):
  File "/var/folders/zr/w_kl9yzd6hsbts7q37qt6d1r0000gn/T/ipykernel_47254/3935025188.py", line 15, in <module>
    session = client.agentic_loop.create_session(
              ^^^^^^^^^^^^^^^^^^^
AttributeError: 'Client' object has no attribute 'agentic_loop'
